# Run one experiment (kaggle)

1. Run the first two cells (they prepare code, data and pretrained weights; safe to re-run).
2. Set `RUN_ID` and `SEED` below (the list shows every ID and whether it is done), then run the rest.

- Code: add the `polypssl` folder (or `polypssl.zip`) as a Kaggle input dataset; enable Internet and a GPU.
- Use **Save Version -> Save & Run All** so the run continues after you close the browser.
- Results: `/kaggle/working/outputs/<ID>__<name>/seed<k>/` = the notebook's Output tab. Download that folder
  (or copy it to `My Drive/polypssl/outputs/`) when the run finishes.

Each run folder ends with `model_final.pth` (kept), `log.csv`, `config.yaml`, `per_image.csv`, `summary.csv`;
`outputs/ALL_RESULTS.csv` collects every finished run. Finished runs are never retrained.

In [ ]:
import os
import subprocess
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

PLATFORM = "kaggle"
WORK = Path("/kaggle/working")
CODE_DIR = WORK / "polypssl"
INPUT_DIR = Path("/kaggle/input")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError("This notebook must run in a Kaggle environment")

candidates = sorted({
    path.parent for path in INPUT_DIR.rglob("train.py")
    if (path.parent / "polypssl").is_dir()
})
if len(candidates) == 1:
    shutil.copytree(candidates[0], CODE_DIR, dirs_exist_ok=True)
elif len(candidates) > 1:
    raise RuntimeError(f"Multiple code roots found in Kaggle inputs: {candidates}")
else:
    archives = sorted(INPUT_DIR.rglob("polypssl.zip"))
    if len(archives) != 1:
        raise FileNotFoundError(
            "Add exactly one code input with polypssl.zip or the extracted project folder")
    with zipfile.ZipFile(archives[0]) as zf:
        names = [PurePosixPath(info.filename.replace("\\", "/")) for info in zf.infolist() if not info.is_dir()]
        if any(path.is_absolute() or ".." in path.parts or
               (path.parts and path.parts[0].endswith(":")) for path in names):
            raise ValueError(f"Unsafe path found in {archives[0]}")
        prefix = PurePosixPath("polypssl")
        if not any(path == prefix / "train.py" for path in names):
            prefix = PurePosixPath(".")
        if not any(path == prefix / "train.py" for path in names):
            raise FileNotFoundError("The code ZIP must contain train.py and the polypssl package")
        for info in zf.infolist():
            path = PurePosixPath(info.filename.replace("\\", "/"))
            if info.is_dir() or "__MACOSX" in path.parts:
                continue
            try:
                relative = path.relative_to(prefix) if str(prefix) != "." else path
            except ValueError:
                continue
            if not relative.parts:
                continue
            target = CODE_DIR.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

os.environ["POLYPSSL_ENV"] = PLATFORM
os.environ["POLYPSSL_DATA"] = "/kaggle/working/data"
os.environ["POLYPSSL_OUT"] = "/kaggle/working/outputs"

if not (CODE_DIR / "train.py").is_file():
    raise FileNotFoundError(f"train.py was not prepared in {CODE_DIR}")
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print(f"[{PLATFORM}] code: {CODE_DIR}")
print(f"[{PLATFORM}] data: {os.environ['POLYPSSL_DATA']}")
print(f"[{PLATFORM}] outputs: {os.environ['POLYPSSL_OUT']}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python setup_env.py

In [ ]:
!python run.py --list --seed 0

## Choose the run

In [ ]:
RUN_ID = "C1"     # e.g. C1, B2, D1, E1  (see the list above and experiments.yaml)
SEED = 0

In [ ]:
!python run.py --run {RUN_ID} --seed {SEED}

## Results

In [ ]:
from IPython.display import Image, display
from polypssl.paths import get_locations

folder = next(get_locations().out.glob(f"{RUN_ID}__*/seed{SEED}"), None)
print("run folder:", folder)
if folder is not None and (folder / "report.txt").exists():
    print((folder / "report.txt").read_text())
    for png in sorted(folder.glob("*.png")):
        display(Image(filename=str(png)))

In [ ]:
!python run.py --results

## Only after a new test set (e.g. PolypGen) is added

Evaluates every finished model again; nothing is retrained (a few minutes per model). Remove the `#` to run it.

In [ ]:
# !python run.py --run all --reevaluate --seed {SEED}